## List Dialogue 
15 jan 2026 (first test)

## Persona
1. **MBTI:** INTJ
    1. รุ่นพี่ใจดีที่มีความรู้พร้อมตอบคำถามที่นักศึกษาหรือคนที่จะสมัครเข้าสาขาอยู่ตลอดแต่ก็แฝงความโหดถ้าใครมาล้ำเกิน ตอบแบบกึ่งวิชาการแบบกันเอง เพื่อความน่าเชื่อถืง
    2. character เป็นแบบมีแผนการรองรับอยู่เสมอฉลาด คิดอย่างเป็นระบบ เป็นนักกลยุทธ์ มีความมุ่งมั่น กระตือรื้อร้นและสามารถทำตามแผนได้
    3. คุยกับคนได้คล่องวิเคราะห์คนออก พร้อมให้ถ่ายทอดองค์ความรู้และตอบคำถามเกี่ยวกับสาขาปัญญาประดิษฐ์และวิทยาการข้อมูลอย่างกระฉับกระเฉงและมั่นใจ

## Library

In [ ]:
#Data & File Handling
import pandas as pd
import io #แบบสำเนา pd
import os #ใช้สั่งการระบบปฏิบัติการ

#System & Display
import asyncio #ใช้รันโปรแกรมแบบ "ไม่รอกัน" (Asynchronous)
from IPython.display import Audio, display #ใช้สร้าง "เครื่องเล่นเสียง" และแสดงผลออกมาบนหน้าจอ

#TTS Engines
import edge_tts
from gtts import gTTS
from pythaitts import TTS


#Post-Process Tune
import pydub 
import torch

Library ที่จัดการเรื่องเสียงอย่าง pydub หรือโมเดล TTS บางตัว ต้องใช้ FFmpeg เป็นเครื่องมือหลังบ้านในการประมวลผลไฟล์มัลติมีเดีย

In [2]:
import subprocess

try:
    version = subprocess.check_output(["ffmpeg", "-version"])
    print("✅ FFmpeg พร้อมใช้งานแล้วบน Mac M4!")
except FileNotFoundError:
    print("❌ ยังหา FFmpeg ไม่เจอ กรุณาเช็คการติดตั้ง brew install ffmpeg")

✅ FFmpeg พร้อมใช้งานแล้วบน Mac M4!


In [3]:
if torch.backends.mps.is_available():
    print("พบ GPU (MPS) บน Mac M4 พร้อมใช้งาน")
else:
    print("😢 ยังใช้แค่ CPU อยู่")

พบ GPU (MPS) บน Mac M4 พร้อมใช้งาน


In [5]:
pip install edge-tts gTTS ipython

Note: you may need to restart the kernel to use updated packages.


In [14]:
# ใช้ os.getcwd() เพื่อเช็คว่าตอนนี้โปรแกรมมองเห็นตัวเองอยู่ที่ไหน
current_path = os.getcwd()
print(f"ตอนนี้คุณอยู่ที่: {current_path}")

dialogue = "Dialogue_ครั้งที่1.csv"
output_dir = "output"

def load_dl(path):
    df = pd.read_csv(path)
    return df

df = load_dl(dialogue)
print("โหลดข้อมูลสำเร็จ! พร้อมส่งต่อให้แต่ละโมเดลแล้วค่ะ")
df.head(3)

ตอนนี้คุณอยู่ที่: /Users/bam/AIDA-chatbot/backend/voice/dialogue
โหลดข้อมูลสำเร็จ! พร้อมส่งต่อให้แต่ละโมเดลแล้วค่ะ


,ID,Category,Scenario,Original Script,TTS Script,Note
0,G01,Greeting,ทักทายครั้งแรกแบบ1,สวัสดีค่า ไอด้านะคะ ยินดีที่ได้รู้จักไอด้าเป็น...,"สวัสดีค่า, ไอด้านะคะ... ยินดีที่ได้รู้จักนะคะ,...",สดใส/เป็นกันเอง
1,G02,Greeting,ทักทายครั้งแรกแบบ2,สวัสดีนะ ยินดีที่ได้รู้จักค่ะ ก่อนอื่นขอแนะนำต...,"สวัสดีนะ, ยินดีที่ได้รู้จักค่ะ. ก่อนอื่นขอแนะน...",มั่นใจ/ฉะฉาน
2,G03,Greeting,ทักทายแบบที่3,Hello สวัสดีนะคะ ไอด้าเอง ยินดีช่วยเหลือค่ะ สง...,"เฮลโล, สวัสดีนะคะ ไอด้าเอง, ยินดีช่วยเหลือค่ะ ...",สุภาพ/พร้อมช่วยเหลือ


## Egde-TTS

In [7]:
async def gen_edge(dataframe, output_path):
    print("📢 เริ่มสร้างเสียงด้วย Edge-TTS (กำลังจูนเสียงใหม่...)")

    for index, row in dataframe.iterrows():
        text_to_speak = row['TTS Script']
        file_id = row['ID']
        filename = f"{file_id}_edge.mp3"
        fullsave_path = os.path.join(output_path, filename)

        # 1. เช็คว่าถ้ามีไฟล์เดิมอยู่แล้ว ให้ลบทิ้งก่อนเพื่อความสดใหม่ (Optional)
        if os.path.exists(fullsave_path):
            os.remove(fullsave_path)

        # 2. ตั้งค่าการปรับจูน 
        target_rate = "+9%" 
        target_pitch = "+10Hz" 

        # 3. ส่งค่าจูนเข้าไปใน Communicate
        communicate = edge_tts.Communicate(
            text_to_speak, 
            "th-TH-PremwadeeNeural", 
            rate=target_rate, 
            pitch=target_pitch
        )
        
        await communicate.save(fullsave_path)
    
        print(f"✅ Re-generated: {file_id} (Rate: {target_rate}, Pitch: {target_pitch})")
        print(f"   Scenario: {row['Scenario']}")
        display(Audio(fullsave_path))


await gen_edge(df, output_dir)

📢 เริ่มสร้างเสียงด้วย Edge-TTS (กำลังจูนเสียงใหม่...)
✅ Re-generated: G01 (Rate: +9%, Pitch: +10Hz)
   Scenario: ทักทายครั้งแรกแบบ1


✅ Re-generated: G02 (Rate: +9%, Pitch: +10Hz)
   Scenario: ทักทายครั้งแรกแบบ2


✅ Re-generated: G03 (Rate: +9%, Pitch: +10Hz)
   Scenario: ทักทายแบบที่3


✅ Re-generated: B01 (Rate: +9%, Pitch: +10Hz)
   Scenario: ถามเรื่องที่ตอบไม่ได้


✅ Re-generated: B02 (Rate: +9%, Pitch: +10Hz)
   Scenario: ถามเรื่องที่ตอบไม่ได้


✅ Re-generated: B03 (Rate: +9%, Pitch: +10Hz)
   Scenario: ถามเรื่องที่ตอบไม่ได้


✅ Re-generated: K01 (Rate: +9%, Pitch: +10Hz)
   Scenario: อธิบายชื่อสาขา


✅ Re-generated: K02 (Rate: +9%, Pitch: +10Hz)
   Scenario: ตอบคำถามเกี่ยวกับสาขา


✅ Re-generated: K03 (Rate: +9%, Pitch: +10Hz)
   Scenario: ตอบคำถามเกี่ยวกับสาขา


✅ Re-generated: K04 (Rate: +9%, Pitch: +10Hz)
   Scenario: ตอบคำถามเกี่ยวกับสาขา


## Google tts

In [12]:
from gtts import gTTS
from pydub import AudioSegment
import os

output_folder = "output_gtts"
if not os.path.exists(output_folder):
    os.makedirs(output_folder)
    print(f"สร้างโฟลเดอร์สำเร็จที่: {output_folder}")

async def gen_gtts(dataframe, output_path):
    print("📢 กำลังเริ่มการสร้างเสียงด้วย Google-TTS (มาตรฐาน)...")

    for index, row in dataframe.iterrows():
        text_to_speak = row['TTS Script']
        file_id = row['ID']
        
        filename = f"{file_id}_google.mp3"
        fullsave_path = os.path.join(output_path, filename)

        # ลบไฟล์เก่าถ้ามีอยู่
        if os.path.exists(filename): 
            os.remove(fullsave_path)
        
        try:
            # เจนเสียง 
            tts = gTTS(text=text_to_speak, lang='th', slow=False) 
            tts.save(fullsave_path)

            print(f"✅ สำเร็จ: {file_id} | Scenario: {row['Scenario']}")
            display(Audio(fullsave_path))
            
        except Exception as e:
            print(f"❌ เกิดข้อผิดพลาดกับ {file_id}: {e}")

# เรียกใช้งาน 
await gen_gtts(df, output_folder)

📢 กำลังเริ่มการสร้างเสียงด้วย Google-TTS (มาตรฐาน)...
✅ สำเร็จ: G01 | Scenario: ทักทายครั้งแรกแบบ1


✅ สำเร็จ: G02 | Scenario: ทักทายครั้งแรกแบบ2


✅ สำเร็จ: G03 | Scenario: ทักทายแบบที่3


✅ สำเร็จ: B01 | Scenario: ถามเรื่องที่ตอบไม่ได้


✅ สำเร็จ: B02 | Scenario: ถามเรื่องที่ตอบไม่ได้


✅ สำเร็จ: B03 | Scenario: ถามเรื่องที่ตอบไม่ได้


✅ สำเร็จ: K01 | Scenario: อธิบายชื่อสาขา


✅ สำเร็จ: K02 | Scenario: ตอบคำถามเกี่ยวกับสาขา


✅ สำเร็จ: K03 | Scenario: ตอบคำถามเกี่ยวกับสาขา


✅ สำเร็จ: K04 | Scenario: ตอบคำถามเกี่ยวกับสาขา


## Pythainlp

ต้องการ python ที่ต่ำกว่าหรือเท่ากับ 3.9 ลงไป กูรันไม่ได้จ้าาาาาา
! ถ้าใช้ window ก็ใช้ Cuda 

In [9]:
!python --version

Python 3.12.9


In [8]:
pip install TTS

ERROR: Ignored the following versions that require a different python version: 0.0.10.2 Requires-Python >=3.6.0,<3.9; 0.0.10.3 Requires-Python >=3.6.0,<3.9; 0.0.11 Requires-Python >=3.6.0,<3.9; 0.0.12 Requires-Python >=3.6.0,<3.9; 0.0.13.1 Requires-Python >=3.6.0,<3.9; 0.0.13.2 Requires-Python >=3.6.0,<3.9; 0.0.14.1 Requires-Python >=3.6.0,<3.9; 0.0.15 Requires-Python >=3.6.0,<3.9; 0.0.15.1 Requires-Python >=3.6.0,<3.9; 0.0.9 Requires-Python >=3.6.0,<3.9; 0.0.9.1 Requires-Python >=3.6.0,<3.9; 0.0.9.2 Requires-Python >=3.6.0,<3.9; 0.0.9a10 Requires-Python >=3.6.0,<3.9; 0.0.9a9 Requires-Python >=3.6.0,<3.9; 0.1.0 Requires-Python >=3.6.0,<3.10; 0.1.1 Requires-Python >=3.6.0,<3.10; 0.1.2 Requires-Python >=3.6.0,<3.10; 0.1.3 Requires-Python >=3.6.0,<3.10; 0.10.0 Requires-Python >=3.7.0,<3.11; 0.10.1 Requires-Python >=3.7.0,<3.11; 0.10.2 Requires-Python >=3.7.0,<3.11; 0.11.0 Requires-Python >=3.7.0,<3.11; 0.11.1 Requires-Python >=3.7.0,<3.11; 0.12.0 Requires-Python >=3.7.0,<3.11; 0.13.0 Requ

In [10]:
try:
    import pythaitts
    print("pythaitts มีอยู่!")
    print("เวอร์ชัน:", pythaitts.__version__ if hasattr(pythaitts, '__version__') else "ไม่พบเวอร์ชัน (ปกติ)")
except ImportError:
    print("❌ pythaitts ยังไม่ได้ติดตั้ง")

pythaitts มีอยู่!
เวอร์ชัน: 0.3.0


In [10]:
try:
    import torch
    print("มี Torch นะจ๊ะ")
    print("version: ", torch.__version__ if hasattr(torch, '__version__') else "ไม้พบเวอร์ชั่น (ปกติ)")
except ImportError:
    print("Torch ยังไม่ได้ติดตั้ง")

มี Torch นะจ๊ะ
version:  2.9.1


In [15]:
output_folder = "output_pythai"
os.makedirs(output_folder, exist_ok=True)
print(f"สร้าง/ใช้โฟลเดอร์: {output_folder}")

async def gen_pythaitts(dataframe, output_path, speaker="Linda"): 
    print(f"📢 กำลังโหลด Model KhanomTan (VITS-based) Mac version (Torch v{torch.__version__} ...")
    try:
     
        # mode='vits' ariational Inference with adversarial learning for end-to-end Text-to-Speech
        # แก้ไข: ใช้ 'khanomtan' ตัวเล็กเพื่อให้หาไฟล์โมเดลเจอ
        tts = pythaitts.TTS(pretrained='khanomtan', device="mps" if torch.backends.mps.is_available() else "cpu")

        for index, row in dataframe.iterrows():
            text_to_speak = row['TTS Script']
            file_id = row['ID']

            filename = f"{file_id}_pythaitts_{speaker}.wav"  
            fullsave_path = os.path.join(output_path, filename)

            if os.path.exists(fullsave_path):
                os.remove(fullsave_path)
                print(f"ลบไฟล์เก่า: {filename}")

            # เรียกใช้ tts แบบเจาะจงพารามิเตอร์ที่ Khanomtan ต้องการ
            tts.tts(
                text=text_to_speak,
                speaker_idx=speaker,       
                # language_idx="th-th", # บางเวอร์ชั่นอาจไม่รองรับพารามิเตอร์นี้ในโหมด VITS ถ้า error ให้คอมเมนต์บรรทัดนี้ออก
                output_file=fullsave_path  
            )
            print(f"✅ สำเร็จ: {file_id} | Speaker: {speaker}")
            display(Audio(fullsave_path, autoplay=False))  # แสดงให้ฟังทันที (autoplay=False เพื่อไม่เล่นอัตโนมัติทุกอัน)

    except Exception as e:
        print(f"❌ เกิดข้อผิดพลาดละเอียด: {type(e).__name__} - {e}")

#  speaker 
await gen_pythaitts(df, output_folder, speaker="Linda")      # Default 
# await gen_pythaitts(df, output_folder, speaker="Tsyncone") # เสียงไทยแท้จาก TSync
# await gen_pythaitts(df, output_folder, speaker="Tsynctwo") # อีกตัวไทยแท้

สร้าง/ใช้โฟลเดอร์: output_pythai
📢 กำลังโหลด Model KhanomTan (VITS-based) Mac version (Torch v2.9.1 ...
❌ เกิดข้อผิดพลาดละเอียด: ModuleNotFoundError - No module named 'TTS'


## F5-TTS-THAI

ต้องการ Numpy <= 2.3 ลงไป 

In [10]:
pip install f5-tts

Note: you may need to restart the kernel to use updated packages.


In [11]:
!pip install git+https://github.com/VYNCX/F5-TTS-THAI.git

  Cloning https://github.com/VYNCX/F5-TTS-THAI.git to /private/var/folders/3j/dn3gq7jj1pg8vq9q_h_3kyfh0000gn/T/pip-req-build-jt46aiaq
  Running command git clone --filter=blob:none --quiet https://github.com/VYNCX/F5-TTS-THAI.git /private/var/folders/3j/dn3gq7jj1pg8vq9q_h_3kyfh0000gn/T/pip-req-build-jt46aiaq
  Resolved https://github.com/VYNCX/F5-TTS-THAI.git to commit 7b0989f5f67378cdea12a117e140c11ac0a9f57d
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done


In [7]:
!pip show -f f5-tts-thai

Name: f5-tts-thai
Version: 1.1.2
Summary: F5-TTS: Open Source Text-to-Speech (TTS) ภาษาไทย — เครื่องมือสร้างเสียงพูดจากข้อความด้วยเทคนิค Flow Matching
Home-page: https://github.com/SWivid/F5-TTS
Author: 
Author-email: 
License: MIT License
Location: /Users/bam/AIDA-chatbot/.venv/lib/python3.12/site-packages
Requires: accelerate, cached_path, click, datasets, deep-translator, ema_pytorch, faster_whisper, gradio, hydra-core, jieba, librosa, matplotlib, numpy, pydantic, pydub, pypinyin, pythainlp, python-crfsuite, safetensors, soundfile, ssg, syllapy, tltk, tomli, torch, torchaudio, torchcodec, torchdiffeq, tqdm, transformers, transformers_stream_generator, vocos, wandb, x_transformers
Required-by: 
Files:
  ../../../bin/f5-tts_finetune-cli
  ../../../bin/f5-tts_finetune-gradio
  ../../../bin/f5-tts_infer-cli
  ../../../bin/f5-tts_infer-gradio
  ../../../bin/f5-tts_webui
  f5_tts/__pycache__/api.cpython-312.pyc
  f5_tts/__pycache__/f5_tts_webui.cpython-312.pyc
  f5_tts/__pycache__/socket_

In [13]:
!pip install "numpy<2.4"

In [11]:
import torch
import soundfile as sf
from f5_tts.model import DiT  # ตัวอย่างการเรียกใช้

# ตรวจสอบการใช้ MPS (GPU Mac)
device = "mps" if torch.backends.mps.is_available() else "cpu"
print(f"🚀 รัน F5-TTS บน: {device}")

# หมายเหตุ: การรัน F5-TTS ครั้งแรกจะมีการโหลด Weight โมเดลขนาดใหญ่ 
# ซึ่ง M4 ของคุณที่มี Unified Memory จะช่วยให้โหลดและรันได้เร็วมาก

/Users/bam/AIDA-chatbot/.venv/lib/python3.12/site-packages/jieba/_compat.py:18: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  import pkg_resources
/Users/bam/AIDA-chatbot/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


🚀 รัน F5-TTS บน: mps


In [16]:
import torchaudio

input_file = "BamtrainF5tts.m4a" 
output_file = "Bamvoice.wav"

# 2. โหลดและบันทึกใหม่เป็น .wav
waveform, sample_rate = torchaudio.load(input_file)
torchaudio.save(output_file, waveform, sample_rate)

print(f"✅ แปลงไฟล์สำเร็จเป็น: {output_file}")

✅ แปลงไฟล์สำเร็จเป็น: Bamvoice.wav


In [13]:
import os
import torch
import soundfile as sf
from f5_tts.api import F5TTS  # <--- ใช้ชื่อนี้ครับ f5_tts
from pythainlp.tokenize import word_tokenize
from IPython.display import Audio, display

# 1. กำหนด Path (Absolute Path)
CURRENT_DIR = os.getcwd() 
REF_AUDIO_PATH = os.path.join(CURRENT_DIR, "Bamvoice.wav")
output_folder = os.path.join(CURRENT_DIR, "output_F5tts_results")

os.makedirs(output_folder, exist_ok=True)

async def gen_f5ttsthai(dataframe, output_path): 
    print(f"📢 กำลังโหลด Model (f5_tts จากค่าย VYNCX)...")
    try:
        device = "mps" if torch.backends.mps.is_available() else "cpu"
        # สร้าง Object จากคลาส F5TTS
        tts = F5TTS(model= "F5-TTS", device=device)
        
        # ตัดคำใน Reference Text
        ref_text_raw = "สวัสดีค่ะนี้เสียงของแบมเอง เราจะใช้เสียงนี้ในการเทสในโมเดลของ F5 ในการเทสระบบน้องไอด้า"
        ref_text_processed = " ".join(word_tokenize(ref_text_raw))

        for index, row in dataframe.iterrows():
            raw_text = row['TTS Script']
            file_id = row['ID']
            text_to_speak = " ".join(word_tokenize(raw_text)) 
            
            filename = f"{file_id}_f5.wav"  
            fullsave_path = os.path.join(output_path, filename)

            print(f"⏳ กำลังประมวลผล: {text_to_speak}")

            wav, sr, _ = tts.infer(
                ref_file=REF_AUDIO_PATH, 
                ref_text=ref_text_processed, 
                gen_text=text_to_speak,
                speed=1.0
            )

            sf.write(fullsave_path, wav)
            print(f"✅ สำเร็จ: {filename}")
            display(Audio(data=wav, autoplay=False))

    except Exception as e:
        print(f"❌ Error: {e}")
        import traceback
        traceback.print_exc()

# เริ่มรัน
await gen_f5ttsthai(df, output_folder)

📢 กำลังโหลด Model (f5_tts จากค่าย VYNCX)...
❌ Error: [Errno 2] No such file or directory: '/Users/bam/AIDA-chatbot/.venv/lib/python3.12/site-packages/f5_tts/configs/F5-TTS.yaml'


Traceback (most recent call last):
  File "/var/folders/3j/dn3gq7jj1pg8vq9q_h_3kyfh0000gn/T/ipykernel_8161/276918955.py", line 20, in gen_f5ttsthai
    tts = F5TTS(model= "F5-TTS", device=device)
          ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/Users/bam/AIDA-chatbot/.venv/lib/python3.12/site-packages/f5_tts/api.py", line 35, in __init__
    model_cfg = OmegaConf.load(str(files("f5_tts").joinpath(f"configs/{model}.yaml")))
                ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/Users/bam/AIDA-chatbot/.venv/lib/python3.12/site-packages/omegaconf/omegaconf.py", line 189, in load
    with io.open(os.path.abspath(file_), "r", encoding="utf-8") as f:
         ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
FileNotFoundError: [Errno 2] No such file or directory: '/Users/bam/AIDA-chatbot/.venv/lib/python3.12/site-packages/f5_tts/configs/F5-TTS.yaml'


## Respond Time

ดูเรื่อง Benchmarking และวัด Latency

สิ่งที่ต้องเก็บข้อมูล (Metrics):

1. Time to First Byte (TTFB): เวลาตั้งแต่ส่ง Text เข้าไปจนเริ่มได้ยินเสียงแรก

2. Generation Time: เวลาทั้งหมดที่ใช้สร้างไฟล์เสียงจนเสร็จ

3. Real-time Factor (RTF): เอา เวลาที่ AI ใช้เจนเสียง หารด้วย ความยาวของเสียงที่ได้ (ถ้า RTF < 1 แสดงว่าเจนเร็วกว่าคนพูดจริง)

/Users/bam/AIDA-chatbot/backend/voice/dialogue
